# 3. Classification (Predicting Popularity Category)


Let m be the median of the column popularity. Create a new binary target variable:

* popularity_binary = 0 if popularity ≤ m

* popularity_binary = 1 if popularity > m

Train a classification model with target popularity_binary.

## Requirements:
* Remove the original popularity column.
* Train a classification model that performs as well as possible.
* Explore more than one modelling approach and justify your final choic

# Imports

In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate

from xgboost import XGBClassifier

# Data Preparation

In [94]:
df = pd.read_csv('tracks2026.csv')
df

,track_id,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5xmO5SbFOiVrRGrMQhL4Jk,44.0,203337,False,0.608,0.553,10,-3.493,1,0.0271,0.3930,0.000000,0.1530,0.541,143.993,3,r-n-b
1,5cF0dROlMOK5uNZtivgu50,83.0,208786,False,0.775,0.613,3,-4.586,0,0.0542,0.1090,0.000023,0.1340,0.797,100.066,4,pop
2,4OQ9XGe11ckizN2EBnNED2,49.0,262373,False,0.797,0.612,2,-9.043,0,0.0313,0.2710,0.000011,0.3140,0.919,118.162,4,synth-pop
3,6Grw9OtoslF9JrDJ6pgsQG,0.0,191733,False,0.582,0.829,9,-3.517,1,0.1930,0.0522,0.000000,0.2640,0.716,90.959,4,indie-pop
4,3fGpNiwYr981n72YY4DZvB,41.0,283706,False,0.705,0.713,4,-6.676,0,0.0437,0.2600,0.000016,0.0499,0.498,129.899,4,synth-pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,4pcpqfSn8tfm6vQMMZPjpM,25.0,266133,False,0.794,0.919,9,-3.037,1,0.0619,0.1540,0.000000,0.2480,0.963,133.550,4,synth-pop
1996,1WM80A5a4xDtlndjqjZQIv,52.0,223236,False,0.667,0.629,0,-8.493,1,0.0324,0.1750,0.116000,0.0915,0.455,115.002,4,synth-pop
1997,4drUfuJw6c9M5cXA8p7upO,0.0,162009,False,0.541,0.753,0,-6.512,1,0.0511,0.0677,0.000000,0.1080,0.572,149.743,4,indie-pop
1998,6ULjJomtdRstnT9BPMAf9d,58.0,120000,False,0.853,0.511,7,-6.451,0,0.4220,0.1880,0.000000,0.2230,0.795,90.171,4,hip-hop


In [95]:
df.describe()

,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,1960.000000,2000.000000,1960.000000,1960.000000,2000.000000,1961.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,1960.000000,2000.000000
mean,39.805102,217806.433000,0.635897,0.632489,5.295500,400.575884,0.637500,0.078466,0.297378,0.030269,0.180715,0.539963,120.040092,3.912000
std,29.245904,56804.759189,0.138298,0.189087,3.567148,18065.717039,0.480842,0.076223,0.279793,0.119946,0.149938,0.236062,28.520528,0.425848
min,0.000000,60000.000000,0.185000,0.090900,0.000000,-21.089000,0.000000,0.022100,0.000007,0.000000,0.009860,0.035900,51.037000,1.000000
25%,1.750000,181210.000000,0.548000,0.507750,2.000000,-8.988000,0.000000,0.034475,0.052475,0.000000,0.093775,0.353000,96.956000,4.000000
50%,45.000000,211346.000000,0.646000,0.644000,5.000000,-6.924000,1.000000,0.047500,0.195500,0.000007,0.121000,0.536000,118.711500,4.000000
75%,65.000000,246069.750000,0.738000,0.780000,8.000000,-5.390000,1.000000,0.084950,0.490250,0.000638,0.221250,0.730000,139.746250,4.000000
max,100.000000,561133.000000,0.953000,0.996000,11.000000,800000.000000,1.000000,0.515000,0.990000,0.962000,0.986000,0.990000,205.895000,5.000000


In [96]:
# This also deals with the null values
df = df[df['loudness'] < 0]

In [97]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df[missing_df["Missing Count"] > 0]

,Missing Count,Missing %


In [98]:
df  = df.dropna()

In [99]:
median_popularity = df['popularity'].median()
median_popularity

np.float64(45.0)

In [100]:
popularity_binary = [0 if x <= median_popularity else 1 for x in df['popularity']]
popularity_binary

[0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,


In [101]:
df.insert(2, 'popularity_binary', popularity_binary)
df

,track_id,popularity,popularity_binary,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5xmO5SbFOiVrRGrMQhL4Jk,44.0,0,203337,False,0.608,0.553,10,-3.493,1,0.0271,0.3930,0.000000,0.1530,0.541,143.993,3,r-n-b
1,5cF0dROlMOK5uNZtivgu50,83.0,1,208786,False,0.775,0.613,3,-4.586,0,0.0542,0.1090,0.000023,0.1340,0.797,100.066,4,pop
2,4OQ9XGe11ckizN2EBnNED2,49.0,1,262373,False,0.797,0.612,2,-9.043,0,0.0313,0.2710,0.000011,0.3140,0.919,118.162,4,synth-pop
3,6Grw9OtoslF9JrDJ6pgsQG,0.0,0,191733,False,0.582,0.829,9,-3.517,1,0.1930,0.0522,0.000000,0.2640,0.716,90.959,4,indie-pop
4,3fGpNiwYr981n72YY4DZvB,41.0,0,283706,False,0.705,0.713,4,-6.676,0,0.0437,0.2600,0.000016,0.0499,0.498,129.899,4,synth-pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,4pcpqfSn8tfm6vQMMZPjpM,25.0,0,266133,False,0.794,0.919,9,-3.037,1,0.0619,0.1540,0.000000,0.2480,0.963,133.550,4,synth-pop
1996,1WM80A5a4xDtlndjqjZQIv,52.0,1,223236,False,0.667,0.629,0,-8.493,1,0.0324,0.1750,0.116000,0.0915,0.455,115.002,4,synth-pop
1997,4drUfuJw6c9M5cXA8p7upO,0.0,0,162009,False,0.541,0.753,0,-6.512,1,0.0511,0.0677,0.000000,0.1080,0.572,149.743,4,indie-pop
1998,6ULjJomtdRstnT9BPMAf9d,58.0,1,120000,False,0.853,0.511,7,-6.451,0,0.4220,0.1880,0.000000,0.2230,0.795,90.171,4,hip-hop


In [102]:
X = df.drop(['track_id', 'popularity_binary', 'popularity'], axis=1)
X

,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,203337,False,0.608,0.553,10,-3.493,1,0.0271,0.3930,0.000000,0.1530,0.541,143.993,3,r-n-b
1,208786,False,0.775,0.613,3,-4.586,0,0.0542,0.1090,0.000023,0.1340,0.797,100.066,4,pop
2,262373,False,0.797,0.612,2,-9.043,0,0.0313,0.2710,0.000011,0.3140,0.919,118.162,4,synth-pop
3,191733,False,0.582,0.829,9,-3.517,1,0.1930,0.0522,0.000000,0.2640,0.716,90.959,4,indie-pop
4,283706,False,0.705,0.713,4,-6.676,0,0.0437,0.2600,0.000016,0.0499,0.498,129.899,4,synth-pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,266133,False,0.794,0.919,9,-3.037,1,0.0619,0.1540,0.000000,0.2480,0.963,133.550,4,synth-pop
1996,223236,False,0.667,0.629,0,-8.493,1,0.0324,0.1750,0.116000,0.0915,0.455,115.002,4,synth-pop
1997,162009,False,0.541,0.753,0,-6.512,1,0.0511,0.0677,0.000000,0.1080,0.572,149.743,4,indie-pop
1998,120000,False,0.853,0.511,7,-6.451,0,0.4220,0.1880,0.000000,0.2230,0.795,90.171,4,hip-hop


In [103]:
y = df['popularity_binary']
y

0       0
1       1
2       1
3       0
4       0
       ..
1995    0
1996    1
1997    0
1998    1
1999    0
Name: popularity_binary, Length: 1960, dtype: int64

In [104]:
preprocess_pipeline = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(drop='first', sparse_output=False), ['track_genre'])
    ],
    remainder=RobustScaler()
).set_output(transform="pandas")

In [105]:
X_transformed = preprocess_pipeline.fit_transform(X)
X_transformed

,categorical__track_genre_indie-pop,categorical__track_genre_pop,categorical__track_genre_r-n-b,categorical__track_genre_synth-pop,remainder__duration_ms,remainder__explicit,remainder__danceability,remainder__energy,remainder__key,remainder__loudness,remainder__mode,remainder__speechiness,remainder__acousticness,remainder__instrumentalness,remainder__liveness,remainder__valence,remainder__tempo,remainder__time_signature
0,0.0,0.0,1.0,0.0,-0.122474,0.0,-0.200000,-0.334252,0.833333,0.953983,0.0,-0.409894,0.450117,-0.010359,0.245710,0.017287,0.590824,-1.0
1,0.0,1.0,0.0,0.0,-0.038365,0.0,0.678947,-0.113866,-0.333333,0.650076,-1.0,0.137304,-0.197140,0.025667,0.097504,0.698138,-0.435742,0.0
2,0.0,0.0,0.0,1.0,0.788783,0.0,0.794737,-0.117539,-0.500000,-0.589184,-1.0,-0.325088,0.172070,0.006185,1.501560,1.022606,-0.012842,0.0
3,1.0,0.0,0.0,0.0,-0.301589,0.0,-0.336842,0.679522,0.666667,0.947310,0.0,2.939929,-0.326591,-0.010359,1.111544,0.482713,-0.648571,0.0
4,0.0,0.0,0.0,1.0,1.118071,0.0,0.310526,0.253444,-0.166667,0.068956,-1.0,-0.074710,0.147000,0.013606,-0.558502,-0.097074,0.261450,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.0,0.0,0.0,1.0,0.846821,0.0,0.778947,1.010101,0.666667,1.080773,0.0,0.292781,-0.094582,-0.010359,0.986739,1.139628,0.346773,0.0
1996,0.0,0.0,0.0,1.0,0.184679,0.0,0.110526,-0.055096,-0.833333,-0.436257,0.0,-0.302877,-0.046721,179.347971,-0.234009,-0.211436,-0.086690,0.0
1997,1.0,0.0,0.0,0.0,-0.760397,0.0,-0.552632,0.400367,-0.833333,0.114556,0.0,0.074710,-0.291265,-0.010359,-0.105304,0.099734,0.725200,0.0
1998,0.0,0.0,0.0,0.0,-1.408831,0.0,1.089474,-0.488522,0.333333,0.131517,-1.0,7.563857,-0.017093,-0.010359,0.791732,0.692819,-0.666986,0.0


In [106]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

# Model Training

In [107]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "Accuracy": "accuracy",
    "F1-score": "f1",
    "Precision": "precision",
    "Recall": "recall",
    "ROC_AUC": "roc_auc",
    "AP": "average_precision"
}

## Random Forest

In [108]:
pipe_rf = Pipeline(steps=[('preprocess', preprocess_pipeline), ('rf', RandomForestClassifier())])
set_config(display="diagram")

In [109]:
pipe_rf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",RobustScaler()
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta

In [125]:
param_grid_rf = {
    'rf__n_estimators': [500, 750, 1000],
    'rf__max_depth': [10, 15, 20]
}

rf_search = GridSearchCV(
    pipe_rf,
    param_grid_rf,
    n_jobs=-1,
    cv=cv,
    scoring=scoring,
    refit="F1-score",
    return_train_score=False
)

rf_search.fit(X_train, y_train)
print(f"Best CV F1 = {rf_search.best_score_:.3f}")
print("Best parameters: ", rf_search.best_params_)

rf_best_params = rf_search.best_params_
rf_best_model = rf_search.best_estimator_
rf_best_cv_f1 = rf_search.best_score_

Best CV F1 = 0.735
Best parameters:  {'rf__max_depth': 15, 'rf__n_estimators': 500}


## XGBoost

In [116]:
pipe_xgb = Pipeline(steps=[('preprocess', preprocess_pipeline), ('xgb', XGBClassifier())])

In [117]:
pipe_xgb

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",RobustScaler()
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [120]:
param_grid_xgb = {
    'xgb__n_estimators': [500, 750, 1000],
    'xgb__max_depth': [10, 15, 20],
    'xgb__learning_rate': [0.01, 0.1]
}

xgb_search = GridSearchCV(
    pipe_xgb,
    param_grid_xgb,
    n_jobs=-1,
    cv=cv,
    scoring=scoring,
    refit="F1-score",
    return_train_score=False
)

xgb_search.fit(X_train, y_train)
print(f"Best CV F1 = {xgb_search.best_score_:.3f}")
print("Best parameters: ", xgb_search.best_params_)

xgb_best_params = xgb_search.best_params_
xgb_best_model = xgb_search.best_estimator_
xgb_best_cv_f1 = xgb_search.best_score_

Best CV F1 = 0.699
Best parameters:  {'xgb__learning_rate': 0.01, 'xgb__max_depth': 15, 'xgb__n_estimators': 750}


# Cross-validation

## Random Forest

In [126]:
res_rf = cross_validate(
        rf_best_model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

In [127]:
for m, vals in res_rf.items():
        print(f"  {m:9s}: mean={np.mean(vals):.3f}, std={np.std(vals):.3f}")

  fit_time : mean=1.543, std=0.030
  score_time: mean=0.123, std=0.005
  test_Accuracy: mean=0.731, std=0.033
  test_F1-score: mean=0.732, std=0.035
  test_Precision: mean=0.723, std=0.037
  test_Recall: mean=0.744, std=0.051
  test_ROC_AUC: mean=0.800, std=0.027
  test_AP  : mean=0.781, std=0.031


In [128]:
feature_names = rf_best_model.named_steps['preprocess'].get_feature_names_out()
importances = rf_best_model.named_steps['rf'].feature_importances_

feature_importance_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print(feature_importance_df)

remainder__acousticness               0.096202
remainder__duration_ms                0.094895
remainder__tempo                      0.091020
remainder__energy                     0.088959
remainder__valence                    0.087527
remainder__speechiness                0.086744
remainder__danceability               0.082508
remainder__loudness                   0.079335
remainder__liveness                   0.074418
remainder__instrumentalness           0.056148
remainder__key                        0.047303
categorical__track_genre_r-n-b        0.038714
categorical__track_genre_pop          0.018230
remainder__mode                       0.013858
remainder__explicit                   0.012942
categorical__track_genre_synth-pop    0.011251
categorical__track_genre_indie-pop    0.011139
remainder__time_signature             0.008807
dtype: float64


## XGBoost

In [121]:
res_xgb = cross_validate(
        xgb_best_model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

In [122]:
for m, vals in res_xgb.items():
        print(f"  {m:9s}: mean={np.mean(vals):.3f}, std={np.std(vals):.3f}")

  fit_time : mean=4.927, std=0.054
  score_time: mean=0.048, std=0.014
  test_Accuracy: mean=0.703, std=0.014
  test_F1-score: mean=0.699, std=0.018
  test_Precision: mean=0.704, std=0.021
  test_Recall: mean=0.696, std=0.037
  test_ROC_AUC: mean=0.782, std=0.022
  test_AP  : mean=0.759, std=0.039


In [123]:
feature_names = xgb_best_model.named_steps['preprocess'].get_feature_names_out()
importances = xgb_best_model.named_steps['xgb'].feature_importances_

feature_importance_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print(feature_importance_df)

categorical__track_genre_r-n-b        0.247459
remainder__explicit                   0.088581
categorical__track_genre_synth-pop    0.079666
categorical__track_genre_indie-pop    0.050951
remainder__acousticness               0.042983
remainder__time_signature             0.042044
remainder__duration_ms                0.041499
remainder__mode                       0.041215
remainder__speechiness                0.041211
remainder__tempo                      0.040081
remainder__energy                     0.039425
remainder__instrumentalness           0.038283
remainder__danceability               0.036746
remainder__key                        0.035469
categorical__track_genre_pop          0.035452
remainder__loudness                   0.034639
remainder__valence                    0.034545
remainder__liveness                   0.029751
dtype: float32
